In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Generate Synthetic Dataset with Names and Account IDs
np.random.seed(42)
n_samples = 1000

# Simple name generator components
first_names = ['James', 'Mary', 'Robert', 'Patricia', 'John', 'Jennifer', 'Michael', 'Linda']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis']

data = {
    'account_id': np.arange(10001, 10001 + n_samples),
    'name': [f"{np.random.choice(first_names)} {np.random.choice(last_names)}" for _ in range(n_samples)],
    'credit_score': np.random.randint(500, 850, n_samples),
    'days_past_due': np.random.choice([30, 60, 90], n_samples),
    'utilization_rate': np.random.uniform(0, 1, n_samples),
    'last_payment_amount': np.random.uniform(0, 500, n_samples),
    'num_historical_delinquencies': np.random.randint(0, 5, n_samples)
}

df = pd.DataFrame(data)

# Risk Logic for outcome (1 = Roll, 0 = Cure)
def determine_outcome(row):
    risk_score = 0
    if row['credit_score'] < 600: risk_score += 0.4
    if row['days_past_due'] >= 60: risk_score += 0.3
    if row['utilization_rate'] > 0.8: risk_score += 0.2
    return 1 if (risk_score + np.random.random() * 0.5) > 0.6 else 0

df['outcome'] = df.apply(determine_outcome, axis=1)

# 2. Build and Train the Model
# Drop non-predictive columns (ID, Name) and target (Outcome)
X = df.drop(['outcome', 'account_id', 'name'], axis=1)
y = df['outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 3. Generate Predictions and Probabilities
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# --- OUTPUT 1: CLASSIFICATION REPORT ---
print("--- MODEL PERFORMANCE (CLASSIFICATION REPORT) ---")
print(classification_report(y_test, y_pred))
print("-" * 50 + "\n")

# --- OUTPUT 2: DETAILED ROLL REPORT ---
# Reconstruct the test dataframe with original metadata
report_df = df.loc[X_test.index].copy()
report_df['roll_probability'] = y_prob
report_df['predicted_action'] = np.where(y_pred == 1, 'ROLL', 'CURE')

# Filter for predicted "Rolls" and sort by risk
roll_report = report_df[report_df['predicted_action'] == 'ROLL'].sort_values(by='roll_probability', ascending=False)

print("--- HIGH-RISK ROLL REPORT (Top 10 Predictions) ---")
cols_to_show = ['name', 'account_id', 'credit_score', 'days_past_due', 'roll_probability']
print(roll_report[cols_to_show].head(10).to_string(index=False))

--- MODEL PERFORMANCE (CLASSIFICATION REPORT) ---
              precision    recall  f1-score   support

           0       0.74      0.73      0.73       107
           1       0.69      0.70      0.70        93

    accuracy                           0.71       200
   macro avg       0.71      0.71      0.71       200
weighted avg       0.72      0.71      0.72       200

--------------------------------------------------

--- HIGH-RISK ROLL REPORT (Top 10 Predictions) ---
             name  account_id  credit_score  days_past_due  roll_probability
   Jennifer Brown       10530           563             60              1.00
      James Brown       10214           544             60              1.00
    James Johnson       10140           561             60              1.00
     James Garcia       10943           591             90              1.00
   Jennifer Davis       10591           585             60              1.00
   Jennifer Smith       10742           571             60